## Example 1: Basic Avro Schema Definition and Data Serialization

In [ ]:
import fastavro
import io
from fastavro import writer, reader, parse_schema

# Example 1: Basic Avro schema definition and data serialization
print("Example 1: Basic Avro Schema Definition and Data Serialization")

try:
    # Define Avro schema
    schema = {
        "type": "record",
        "name": "User",
        "fields": [
            {"name": "name", "type": "string"},
            {"name": "age", "type": "int"},
            {"name": "email", "type": "string"}
        ]
    }
    
    # Parse schema
    parsed_schema = parse_schema(schema)
    print(f"Schema parsed successfully: {parsed_schema['name']}")
    
    # Sample data
    records = [
        {"name": "Alice", "age": 30, "email": "alice@example.com"},
        {"name": "Bob", "age": 25, "email": "bob@example.com"}
    ]
    
    # Serialize data to bytes
    bytes_writer = io.BytesIO()
    fastavro.writer(bytes_writer, parsed_schema, records)
    serialized_data = bytes_writer.getvalue()
    
    print(f"Data serialized successfully: {len(serialized_data)} bytes")
    print(f"Records serialized: {len(records)}")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fastavro example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Avro Data Deserialization and Reading

In [ ]:
import fastavro
import io
from fastavro import writer, reader, parse_schema

# Example 2: Avro data deserialization and reading
print("Example 2: Avro Data Deserialization and Reading")

try:
    # Create schema and data
    schema = {
        "type": "record",
        "name": "Product",
        "fields": [
            {"name": "id", "type": "int"},
            {"name": "name", "type": "string"},
            {"name": "price", "type": "double"}
        ]
    }
    
    parsed_schema = parse_schema(schema)
    
    # Sample product data
    products = [
        {"id": 1, "name": "Laptop", "price": 999.99},
        {"id": 2, "name": "Mouse", "price": 25.50},
        {"id": 3, "name": "Keyboard", "price": 75.00}
    ]
    
    # Serialize data
    bytes_writer = io.BytesIO()
    fastavro.writer(bytes_writer, parsed_schema, products)
    serialized_data = bytes_writer.getvalue()
    
    # Deserialize data
    bytes_reader = io.BytesIO(serialized_data)
    avro_reader = fastavro.reader(bytes_reader)
    
    print("Deserialized products:")
    for record in avro_reader:
        print(f"ID: {record['id']}, Name: {record['name']}, Price: ${record['price']:.2f}")
    
    print(f"Successfully read {len(products)} records")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fastavro example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Complex Schema with Nested Types and Arrays

In [ ]:
import fastavro
import io
from fastavro import writer, reader, parse_schema

# Example 3: Complex schema with nested types and arrays
print("Example 3: Complex Schema with Nested Types and Arrays")

try:
    # Define complex schema with nested record and arrays
    schema = {
        "type": "record",
        "name": "Order",
        "fields": [
            {"name": "order_id", "type": "string"},
            {
                "name": "customer",
                "type": {
                    "type": "record",
                    "name": "Customer",
                    "fields": [
                        {"name": "name", "type": "string"},
                        {"name": "email", "type": "string"}
                    ]
                }
            },
            {
                "name": "items",
                "type": {
                    "type": "array",
                    "items": {
                        "type": "record",
                        "name": "Item",
                        "fields": [
                            {"name": "product", "type": "string"},
                            {"name": "quantity", "type": "int"},
                            {"name": "price", "type": "double"}
                        ]
                    }
                }
            },
            {"name": "total", "type": "double"}
        ]
    }
    
    parsed_schema = parse_schema(schema)
    
    # Complex nested data
    orders = [{
        "order_id": "ORD-2024-001",
        "customer": {
            "name": "John Doe",
            "email": "john@example.com"
        },
        "items": [
            {"product": "Laptop", "quantity": 1, "price": 999.99},
            {"product": "Mouse", "quantity": 2, "price": 25.50}
        ],
        "total": 1050.99
    }]
    
    # Serialize and deserialize complex data
    bytes_writer = io.BytesIO()
    fastavro.writer(bytes_writer, parsed_schema, orders)
    
    bytes_reader = io.BytesIO(bytes_writer.getvalue())
    for order in fastavro.reader(bytes_reader):
        print(f"Order ID: {order['order_id']}")
        print(f"Customer: {order['customer']['name']} ({order['customer']['email']})")
        print(f"Items: {len(order['items'])} items, Total: ${order['total']:.2f}")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fastavro example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Schema Evolution and Compatibility

In [ ]:
import fastavro
import io
from fastavro import writer, reader, parse_schema

# Example 4: Schema evolution and compatibility
print("Example 4: Schema Evolution and Compatibility")

try:
    # Original schema v1
    schema_v1 = {
        "type": "record",
        "name": "Employee",
        "fields": [
            {"name": "id", "type": "int"},
            {"name": "name", "type": "string"},
            {"name": "department", "type": "string"}
        ]
    }
    
    # Evolved schema v2 with new optional field
    schema_v2 = {
        "type": "record",
        "name": "Employee",
        "fields": [
            {"name": "id", "type": "int"},
            {"name": "name", "type": "string"},
            {"name": "department", "type": "string"},
            {"name": "salary", "type": ["null", "double"], "default": None}  # Optional field
        ]
    }
    
    # Write data with v1 schema
    employees_v1 = [
        {"id": 1, "name": "Alice", "department": "Engineering"},
        {"id": 2, "name": "Bob", "department": "Marketing"}
    ]
    
    bytes_writer = io.BytesIO()
    fastavro.writer(bytes_writer, parse_schema(schema_v1), employees_v1)
    
    # Read data with evolved schema v2 (backward compatibility)
    bytes_reader = io.BytesIO(bytes_writer.getvalue())
    
    print("Reading v1 data with v2 schema (backward compatibility):")
    for employee in fastavro.reader(bytes_reader):
        print(f"ID: {employee['id']}, Name: {employee['name']}")
        print(f"Department: {employee['department']}, Salary: {employee.get('salary', 'Not set')}")
    
    # Write new data with v2 schema
    employees_v2 = [
        {"id": 3, "name": "Carol", "department": "Finance", "salary": 75000.0}
    ]
    
    bytes_writer_v2 = io.BytesIO()
    fastavro.writer(bytes_writer_v2, parse_schema(schema_v2), employees_v2)
    
    print("\nReading v2 data:")
    bytes_reader_v2 = io.BytesIO(bytes_writer_v2.getvalue())
    for employee in fastavro.reader(bytes_reader_v2):
        print(f"ID: {employee['id']}, Name: {employee['name']}, Salary: ${employee['salary']:.2f}")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fastavro example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: File-based Avro Operations and Validation

In [ ]:
import fastavro
import tempfile
import os
from fastavro import writer, reader, parse_schema, validate

# Example 5: File-based Avro operations and validation
print("Example 5: File-based Avro Operations and Validation")

try:
    # Define schema for event logging
    event_schema = {
        "type": "record",
        "name": "Event",
        "fields": [
            {"name": "timestamp", "type": "long"},
            {"name": "event_type", "type": "string"},
            {"name": "user_id", "type": "string"},
            {"name": "metadata", "type": {"type": "map", "values": "string"}}
        ]
    }
    
    parsed_schema = parse_schema(event_schema)
    
    # Sample event data
    events = [
        {
            "timestamp": 1640995200000,
            "event_type": "login",
            "user_id": "user123",
            "metadata": {"ip": "192.168.1.1", "browser": "Chrome"}
        },
        {
            "timestamp": 1640995260000,  # 1 minute later
            "event_type": "page_view",
            "user_id": "user123",
            "metadata": {"page": "/dashboard", "referrer": "/login"}
        }
    ]
    
    # Validate data against schema
    print("Validating data against schema...")
    for i, event in enumerate(events):
        is_valid = fastavro.validate(event, parsed_schema)
        print(f"Event {i+1} validation: {'✓ Valid' if is_valid else '✗ Invalid'}")
    
    # Write to temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix='.avro') as temp_file:
        temp_path = temp_file.name
        
        # Write events to file
        with open(temp_path, 'wb') as out:
            fastavro.writer(out, parsed_schema, events)
        
        print(f"\nEvents written to file: {temp_path}")
        print(f"File size: {os.path.getsize(temp_path)} bytes")
        
        # Read events from file
        with open(temp_path, 'rb') as avro_file:
            avro_reader = fastavro.reader(avro_file)
            schema_from_file = avro_reader.writer_schema
            
            print(f"\nSchema from file: {schema_from_file['name']}")
            print("Events from file:")
            
            for event in avro_reader:
                print(f"  {event['event_type']} by {event['user_id']} at {event['timestamp']}")
        
        # Clean up
        os.unlink(temp_path)
        print("Temporary file cleaned up")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"fastavro example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e